# Code 2.7.4: Gradient checking with finite differences

Computes centered finite-difference gradients for every parameter and compares them with the analytic gradients of Code 2.7.2 on a small 2-5-2 network.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 2.7.2
import numpy as np

def init_params(n_in, n_hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(1 / n_hidden), (n_hidden, n_out)),
        "b2": np.zeros(n_out),
    }

def forward(P, X):
    Z1 = X @ P["W1"] + P["b1"]          # (B, H)   bias broadcasts over rows
    H = np.maximum(0, Z1)               # (B, H)
    Z2 = H @ P["W2"] + P["b2"]          # (B, C)   logits
    return Z2, (X, Z1, H)

def softmax_cross_entropy(Z, y):
    """Mean cross-entropy of integer labels y under softmax(Z); also returns dL/dZ."""
    Zs = Z - Z.max(axis=1, keepdims=True)        # stability: subtract the row max
    logsumexp = np.log(np.exp(Zs).sum(axis=1, keepdims=True))
    logp = Zs - logsumexp                        # log-softmax, (B, C)
    B = len(y)
    loss = -logp[np.arange(B), y].mean()
    dZ = np.exp(logp)                            # probabilities
    dZ[np.arange(B), y] -= 1.0                   # p - onehot(y)
    return loss, dZ / B

def backward(P, cache, dZ2):
    X, Z1, H = cache
    grads = {}
    grads["W2"] = H.T @ dZ2                      # (H, B) @ (B, C) -> (H, C)
    grads["b2"] = dZ2.sum(axis=0)                # (C,)
    dH = dZ2 @ P["W2"].T                         # (B, C) @ (C, H) -> (B, H)
    dZ1 = dH * (Z1 > 0)                          # ReLU passes gradient where Z1 > 0
    grads["W1"] = X.T @ dZ1                      # (D, B) @ (B, H) -> (D, H)
    grads["b1"] = dZ1.sum(axis=0)                # (H,)
    for k in P:
        assert grads[k].shape == P[k].shape, k   # every gradient matches its parameter
    return grads


In [ ]:
def numeric_grads(P, X, y, eps=1e-5):
    num = {}
    for k in P:
        num[k] = np.zeros_like(P[k])
        for idx in np.ndindex(P[k].shape):
            old = P[k][idx]
            P[k][idx] = old + eps
            loss_plus, _ = softmax_cross_entropy(forward(P, X)[0], y)
            P[k][idx] = old - eps
            loss_minus, _ = softmax_cross_entropy(forward(P, X)[0], y)
            P[k][idx] = old                              # restore
            num[k][idx] = (loss_plus - loss_minus) / (2 * eps)
    return num

def rel_error(a, n):
    return np.linalg.norm(a - n) / max(np.linalg.norm(a) + np.linalg.norm(n), 1e-12)

rng = np.random.default_rng(1)
Xc, yc = rng.normal(size=(20, 2)), rng.integers(0, 2, size=20)
P = init_params(2, 5, 2, seed=3)
P["b1"] = rng.normal(0, 0.5, 5)                        # make some ReLUs inactive
Z, cache = forward(P, Xc)
_, dZ = softmax_cross_entropy(Z, yc)
analytic = backward(P, cache, dZ)
numeric = numeric_grads(P, Xc, yc)
for k in P:
    print(f"{k}: relative error {rel_error(analytic[k], numeric[k]):.1e}")


**Expected output**

Output:

```text
W1: relative error 9.1e-11
b1: relative error 4.8e-11
W2: relative error 3.4e-11
b2: relative error 5.5e-12
```
